# Task 3 — Define Input Schema and Validate

Defines a schema for the cleaned freeCodeCamp dataset and validates every row against it. Passing rows go to `data/interim/validated.csv`; failing rows go to `data/interim/rejected.csv` with a reason.

## Imports

In [1]:
import pandas as pd
import re

## Load the cleaned data

In [2]:
df = pd.read_csv("../data/interim/cleaned.csv")
print(f"Loaded {len(df)} rows")
df.head()

Loaded 450 rows


,source,category,title,author,publication_date,description,url,topic,matched_keywords,publication_datetime
0,freeCodeCamp,#shadcn ui,How to Build an AI Chat App Interface With the...,Vaibhav Gupta,2026-09-11,Every other AI product you open today has the ...,https://www.freecodecamp.org/news/how-to-build...,AI,none,2026-09-11 16:21:29.864000+00:00
1,freeCodeCamp,#Artificial Intelligence,How to Build a Self-Evaluating AI System: Auto...,Jude Otine,2026-09-11,So you shipped your AI feature and it works in...,https://www.freecodecamp.org/news/build-a-self...,AI,"artificial intelligence, llm",2026-09-11 15:24:04.941000+00:00
2,freeCodeCamp,#Security,How AI Is Changing Malware Detection: From Tra...,Manish Shivanandhan,2026-09-11,Malware used to be simple to describe. A virus...,https://www.freecodecamp.org/news/how-ai-is-ch...,AI,none,2026-09-11 15:22:46.931000+00:00
3,freeCodeCamp,#AI,How to Build an AI Chatbot with Gemini and Ver...,Johnson Samuel,2026-09-07,"A couple of months back, I built a chatbot app...",https://www.freecodecamp.org/news/how-to-build...,AI,none,2026-09-07 22:35:39.060000+00:00
4,freeCodeCamp,#AI,How AI Receptionists Work: The Architecture Be...,Manish Shivanandhan,2026-09-04,An AI receptionist may sound simple from the o...,https://www.freecodecamp.org/news/how-ai-recep...,AI,none,2026-09-04 20:07:46.216000+00:00


## Schema definition

| Column | Data Type | Nullable | Allowed Values / Range |
|---|---|---|---|
| source | string | N | equals "freeCodeCamp" |
| title | string | N | non-empty |
| url | string | N | starts with "https://" |
| publication_date | date | N | valid date |
| topic | string | N | one of: AI, Cloud, Data Science |
| author | string | N | non-empty (placeholder "No author" counts as present) |
| description | string | Y | — |
| category | string | Y | — |
| matched_keywords | string | Y | — |

## Validation logic

In [3]:
VALID_TOPICS = {"AI", "Cloud", "Data Science"}

def validate_row(row):
    reasons = []

    if row["source"] != "freeCodeCamp":
        reasons.append("source is not 'freeCodeCamp'")

    if pd.isna(row["title"]) or str(row["title"]).strip() == "":
        reasons.append("title is empty")

    if pd.isna(row["url"]) or not str(row["url"]).startswith("https://"):
        reasons.append("url missing or does not start with https://")

    if pd.isna(row["publication_date"]) or str(row["publication_date"]).strip() == "":
        reasons.append("publication_date is missing/invalid")

    if row["topic"] not in VALID_TOPICS:
        topic_value = row["topic"]
        reasons.append(f"topic '{topic_value}' not in {VALID_TOPICS}")

    if pd.isna(row["author"]) or str(row["author"]).strip() == "":
        reasons.append("author is empty")

    return reasons

## Run validation

In [4]:
validated_rows = []
rejected_rows = []

for idx, row in df.iterrows():
    reasons = validate_row(row)
    if reasons:
        rejected_row = row.to_dict()
        rejected_row["rejection_reason"] = "; ".join(reasons)
        rejected_rows.append(rejected_row)
    else:
        validated_rows.append(row.to_dict())

validated_df = pd.DataFrame(validated_rows)
rejected_df = pd.DataFrame(rejected_rows)

print(f"Validated: {len(validated_df)} rows")
print(f"Rejected: {len(rejected_df)} rows")

if len(rejected_df) > 0:
    print(f"\nRejection reasons breakdown:")
    print(rejected_df["rejection_reason"].value_counts())

Validated: 450 rows
Rejected: 0 rows


## Preview rejected rows (if any)

In [5]:
rejected_df.head() if len(rejected_df) > 0 else print("No rejected rows — everything passed validation.")

No rejected rows — everything passed validation.


## Save results

In [6]:
validated_df.to_csv("../data/interim/validated.csv", index=False)
rejected_df.to_csv("../data/interim/rejected.csv", index=False)

print(f"Saved {len(validated_df)} rows to data/interim/validated.csv")
print(f"Saved {len(rejected_df)} rows to data/interim/rejected.csv")

pass_rate = len(validated_df) / len(df) * 100
print(f"\nPass rate: {pass_rate:.1f}%")

Saved 450 rows to data/interim/validated.csv
Saved 0 rows to data/interim/rejected.csv

Pass rate: 100.0%
